# 3D U-Net Baseline — Seed Reproducibility Run

**Purpose:** Multi-seed reproducibility experiment for the 3D U-Net 2ch baseline.
Run this notebook 3 times, changing `SEED` in the CONFIG cell: **42 → 123 → 999**.

**Reference result (uncontrolled seed):** 3D U-Net 2ch = 0.6735 Dice (locked 2026-05-24).
The 3 seeded runs will report mean ± std over n=3 controlled seeds. Used alongside FADC-Bottleneck and FADC-Deep seed runs to compute paired-seed comparisons.

**Model:** UNet3D (22.5M params, no FADC modules)
**Dataset:** MAMA-MIA (1200 train / 306 val), 2-channel input (pre + post contrast)
**Patch size:** 128×128×64 | **Epochs:** 100 | **Warmup:** 5
**GPU:** Kaggle T4 x2 | cudnn.deterministic=True (≈10–20% slower per epoch)

**IMPORTANT:** Download `best_model.pth`, `train_log.json`, `meta.json` from `OUTPUT_DIR` immediately after the run finishes — `/kaggle/working` wipes on session close.

In [ ]:
# ────────────────────────────────────────────────
# CONFIGURATION — edit SEED before each run
# ────────────────────────────────────────────────
# 3D U-Net baseline seed reproducibility experiment (3 runs total).
# Change SEED to 42, then 123, then 999 — re-run the whole notebook for each.

SEED = 42  # <<< CHANGE TO 123 AND 999 FOR THE OTHER TWO RUNS

DATA_ROOT    = "/kaggle/input/datasets/bharathvemurik/mama-mia-preprocessed-cache-2ch"
OUTPUT_DIR   = f"/kaggle/working/outputs/unet3d_2ch_100ep_s{SEED}"
CODE_DIR     = "/kaggle/working/FADC-3D"

EPOCHS       = 100
BATCH_SIZE   = 2
NUM_WORKERS  = 4
PATCH_SIZE   = [128, 128, 64]
WARMUP       = 5

RESUME_FROM  = ""

# 2-channel preprocessed cache (pre + post contrast).
# Slug: bharathvemurik/mama-mia-preprocessed-cache-2ch
PREPROCESSED_CACHE_DIR = "/kaggle/input/datasets/bharathvemurik/mama-mia-preprocessed-cache-2ch"

print(f"SEED       : {SEED}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

In [ ]:
# ────────────────────────────────────────────────
# 1. INSTALL DEPENDENCIES
# ────────────────────────────────────────────────
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "monai",
    "--upgrade-strategy", "only-if-needed", "-q"
], check=True)

import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
print("Dependencies ready.")

In [ ]:
# ────────────────────────────────────────────────
# 2. CLONE / UPDATE CODE FROM GITHUB
# ────────────────────────────────────────────────
import os

if os.path.exists(CODE_DIR):
    print("Repo already exists — pulling latest...")
    os.system(f"git -C {CODE_DIR} pull")
else:
    os.system(f"git clone https://github.com/Vemuri-BK/FADC-3D.git {CODE_DIR}")
    print("Repo cloned.")

sys.path.insert(0, CODE_DIR)
print(f"Code path: {CODE_DIR}")

In [ ]:
# ────────────────────────────────────────────────
# 3. VERIFY GPU
# ────────────────────────────────────────────────
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU             : {gpu.name}")
    print(f"VRAM            : {gpu.total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU — enable GPU T4 x2 in Settings → Accelerator")

In [ ]:
# ────────────────────────────────────────────────
# 4. SANITY-CHECK 2-CHANNEL CACHE
#    Verify .npz files have shape (2, H, W, D) before launching training.
# ────────────────────────────────────────────────
import os, numpy as np
from pathlib import Path

assert PREPROCESSED_CACHE_DIR, "PREPROCESSED_CACHE_DIR is empty — set it in the config cell."
cache_path = Path(PREPROCESSED_CACHE_DIR)
assert cache_path.exists(), f"Cache path not found: {cache_path}"

train_npzs = sorted((cache_path / "train").glob("*.npz"))
val_npzs   = sorted((cache_path / "val").glob("*.npz"))
print(f"Train .npz files : {len(train_npzs)}")
print(f"Val   .npz files : {len(val_npzs)}")
assert len(train_npzs) > 0 and len(val_npzs) > 0, "No .npz files found in train/ or val/ subdirs."

sample_paths = [train_npzs[0], train_npzs[len(train_npzs)//2], train_npzs[-1], val_npzs[0]]
for p in sample_paths:
    d = np.load(p)
    img_shape = d["image"].shape
    lbl_shape = d["label"].shape
    print(f"  {p.name:35s}  image={img_shape}  label={lbl_shape}  img_dtype={d['image'].dtype}")
    assert img_shape[0] == 2, (
        f"FATAL: {p.name} has {img_shape[0]} channels, expected 2."
    )
    assert lbl_shape[0] == 1, f"Label channel mismatch in {p.name}: {lbl_shape}"

print("\nAll sampled cases are 2-channel — cache looks good. Safe to launch training.")

In [ ]:
# ────────────────────────────────────────────────
# 5. RUN TRAINING — 3D U-Net baseline (seed run)
# ────────────────────────────────────────────────
import os, subprocess, sys
os.makedirs(OUTPUT_DIR, exist_ok=True)

train_script = os.path.join(CODE_DIR, "training", "train_centralized.py")

cmd = [
    sys.executable, "-u", train_script,
    "--model",          "unet3d",
    "--data_root",      DATA_ROOT,
    "--output_dir",     OUTPUT_DIR,
    "--epochs",         str(EPOCHS),
    "--batch_size",     str(BATCH_SIZE),
    "--num_workers",    str(NUM_WORKERS),
    "--patch_size",     str(PATCH_SIZE[0]), str(PATCH_SIZE[1]), str(PATCH_SIZE[2]),
    "--warmup_epochs",  str(WARMUP),
    "--seed",           str(SEED),
]

if RESUME_FROM:
    cmd += ["--resume", RESUME_FROM]

if PREPROCESSED_CACHE_DIR:
    cmd += ["--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR]

print("Command:", " ".join(cmd))
print("=" * 60)

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
while True:
    chunk = process.stdout.read(512)
    if not chunk:
        break
    sys.stdout.write(chunk.decode("utf-8", errors="replace"))
    sys.stdout.flush()
process.wait()
print(f"\nExit code: {process.returncode}")

In [ ]:
# ────────────────────────────────────────────────
# 6. PLOT TRAINING CURVES
# ────────────────────────────────────────────────
import json
import matplotlib.pyplot as plt

BASELINE_REF_DICE = 0.6735  # locked uncontrolled-seed baseline (2026-05-24)
PAPER_DICE        = 0.762   # MAMA-MIA paper nnU-Net reference

log_path = os.path.join(OUTPUT_DIR, "train_log.json")

if not os.path.exists(log_path):
    print("No training log found yet.")
else:
    with open(log_path) as f:
        log = json.load(f)

    epochs     = [e["epoch"]    for e in log]
    losses     = [e["loss"]     for e in log]
    val_epochs = [e["epoch"]    for e in log if "val_dice" in e]
    val_dices  = [e["val_dice"] for e in log if "val_dice" in e]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, losses, color="steelblue", linewidth=1.5)
    ax1.set_title("Training Loss", fontsize=13)
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.grid(True, alpha=0.3)

    ax2.plot(val_epochs, val_dices, color="darkorange", linewidth=1.5, marker="o", markersize=4)
    ax2.axhline(y=BASELINE_REF_DICE, color="green", linestyle="--", linewidth=1.5,
                label=f"Baseline reference ({BASELINE_REF_DICE:.4f})")
    ax2.axhline(y=PAPER_DICE, color="red", linestyle="--", linewidth=1,
                label=f"MAMA-MIA paper ({PAPER_DICE:.3f})")
    ax2.set_title("Validation Dice", fontsize=13)
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Dice Score")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    if val_dices:
        best = max(val_dices)
        ax2.set_title(f"Validation Dice  (seed={SEED}, best: {best:.4f})", fontsize=13)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150)
    plt.show()
    print(f"Seed             : {SEED}")
    print(f"Epochs completed : {len(log)}")
    if val_dices:
        print(f"Best Val Dice    : {max(val_dices):.4f}")
        print(f"Reference baseline: {BASELINE_REF_DICE:.4f}")
        print(f"Delta vs reference: {max(val_dices) - BASELINE_REF_DICE:+.4f}")